[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke05-agentisk-ai/03_crewai_fordypning_rehabilitering.ipynb)


# Fordypning: CrewAI i rehabilitering

## Laeringsmaal

- Se hvordan et multi-agent system kan organiseres som et lite tverrfaglig team.
- Forsta hvordan roller, oppgaver og verktoy kan deles opp i et rehabiliteringscase.
- Reflektere over hvorfor menneskelig kontroll fortsatt er nodvendig.
- Se hvordan en lett Gemini-modell kan brukes i en undervisningsdemo med syntetiske data.

> **Merk:** Dette er pedagogisk materiale. Eksemplet er laget for laering og refleksjon, ikke for klinisk beslutningsstotte.

Denne notebooken er et frivillig fordypningsspor til uke 05. Den bygger videre pa skillet mellom chatbot, workflow og agent, og viser hvordan et lite `CrewAI`-oppsett kan brukes til a simulere et tverrfaglig rehabiliteringsteam rundt et syntetisk case.


## Installering og setup

Hvis du kjorer denne notebooken i Google Colab, er den enkleste oppskriften vanligvis:

- installer `crewai`
- legg inn `GOOGLE_API_KEY`
- bruk en lett Gemini-modell, for eksempel `gemini/gemini-2.0-flash`

Denne notebooken bruker syntetiske data, sa en gratis eller rimelig Gemini-modell er som regel tilstrekkelig. Bruk ikke ekte pasientopplysninger i et slikt oppsett.


### Korte steg i Colab

Hvis du vil teste demoen i Colab, kan du gjore dette:

1. Apne notebooken i Colab.
2. Kjor installasjons- og setup-cellen.
3. Lim inn `GOOGLE_API_KEY` nar du blir spurt.
4. Kjor cellene for case, agenter og `CrewAI`.

Hvis alt er satt opp riktig, vil du fa et strukturert undervisningsutkast fra det tverrfaglige rehabiliteringsteamet.


## Hva skal eksemplet vise?

I de forrige notebookene sa vi at agentisk AI blir interessant nar en oppgave krever flere steg, flere perspektiver eller bruk av verktoy underveis. Her tar vi det ett steg videre:

- En agent vurderer funksjon og mobilitet.
- En agent vurderer kommunikasjon og informasjonstilpasning.
- En agent vurderer egenomsorg og hjemmeoppfolging.
- En koordinator samler alt i en strukturert plan.

Poenget er ikke at systemet skal overta faglige vurderinger. Poenget er a se hvordan et multi-agent oppsett kan organisere arbeidsdeling, struktur og sporbarhet.

For a kjoere koden trenger du vanligvis en installert `crewai`-pakke og tilgang til en Gemini API-nokkel. I setup-cellen under sjekker vi om `GOOGLE_API_KEY` finnes, og hvis du bruker Colab kan du lime den inn direkte. Vi bruker en lett Gemini-modell som standard for at demoen skal vaere enkel a kjoere med syntetiske data.


In [ ]:
# Eventuell installasjon i Colab eller ny notebook-kjerne:
# %pip install -q crewai

import os
from getpass import getpass

# Standardmodell for en lett undervisningsdemo.
DEFAULT_GEMINI_MODEL = "gemini/gemini-2.0-flash"

# 1. Proev aa lese API-nokkelen fra miljoet.
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 2. Hvis vi kjoerer i Colab og nokkelen ikke finnes, kan brukeren lime den inn.
if not GOOGLE_API_KEY:
    try:
        import google.colab  # type: ignore
        GOOGLE_API_KEY = getpass("Lim inn GOOGLE_API_KEY: ")
        if GOOGLE_API_KEY:
            os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    except ImportError:
        pass

# 3. Velg modell. Du kan overstyre denne via miljoet med GEMINI_MODEL.
GEMINI_MODEL = os.getenv("GEMINI_MODEL", DEFAULT_GEMINI_MODEL)

print("Valgt modell:", GEMINI_MODEL)
print("GOOGLE_API_KEY funnet:", bool(GOOGLE_API_KEY))

if not GOOGLE_API_KEY:
    print("Sett GOOGLE_API_KEY i miljoet, eller lim den inn hvis du bruker Colab.")


In [ ]:
from pprint import pprint

# Syntetisk rehabiliteringscase
case_text = """
Pasient, 78 ar, etter hjerneslag.
Bor alene. Redusert gangfunksjon og usikker balanse.
Glemmer iblant medisiner. Trenger hjelp til handling og praktiske oppgaver.
Mal: kunne bo trygt hjemme og mestre daglige aktiviteter best mulig.
"""


def case_reader(text: str) -> str:
    return text.strip()


def rehab_checklist_tool() -> str:
    return """
    Rehabiliteringssjekkliste:
    - mobilitet og balanse
    - ADL-funksjon
    - kommunikasjon og informasjonstilpasning
    - medikamenthandtering
    - hjemmeoppfolging
    - parorende og samhandling
    """.strip()


def local_guideline_lookup(topic: str) -> str:
    knowledge = {
        "fallrisiko": "Ved redusert balanse bor tiltak for fallforebygging vurderes.",
        "egenomsorg": "Ved nedsatt egenmestring bor behov for praktisk bistand vurderes.",
        "medikamenter": "Ved usikker legemiddelhandtering bor oppfolging og kontrollrutiner avklares.",
    }
    return knowledge.get(topic.lower(), "Ingen direkte match i kunnskapsbasen.")


def uncertainty_flagger(text: str) -> str:
    return {
        "undervisningsutkast": True,
        "menneskelig_kontroll": "Obligatorisk for alle klinisk relevante forslag.",
        "resultat": text,
    }


print("CASE")
print(case_reader(case_text))

print("\nSJEKKLISTE")
print(rehab_checklist_tool())

print("\nEKSEMPEL PAA LOKALT OPPSLAG")
pprint(
    {
        "fallrisiko": local_guideline_lookup("fallrisiko"),
        "egenomsorg": local_guideline_lookup("egenomsorg"),
        "medikamenter": local_guideline_lookup("medikamenter"),
    }
)


## Fra enkel agent til crew

I notebook `02_agentisk_ai_i_helse.ipynb` sa vi at en enkel agent kan velge verktoy, hente kontekst og lage et forslag. I et `CrewAI`-oppsett fordeles samme type oppgave pa flere roller.

I dette mini-eksemplet bruker vi fire agenter:

- `Funksjonsagent`: vurderer mobilitet, ADL og fallrisiko.
- `Kommunikasjonsagent`: vurderer behov for tilpasset informasjon.
- `Omsorgsagent`: vurderer egenomsorg, medisiner og hjemmeoppfolging.
- `Koordinatoragent`: samler delvurderingene i en strukturert plan.

Legg merke til at agentene ikke skal stille diagnose eller ta selvstendige kliniske beslutninger. De skal lage et strukturert utkast som senere ma vurderes av et menneske.


In [ ]:
from crewai import Agent, Task, Crew, Process, LLM


if not GOOGLE_API_KEY:
    raise ValueError("Sett GOOGLE_API_KEY for aa kjoere denne demoen mot Gemini.")


print(f"Kjoerer CrewAI med modell: {GEMINI_MODEL}")

llm = LLM(
    model=GEMINI_MODEL,
    temperature=0.2,
)


funksjonsagent = Agent(
    role="Funksjonsagent",
    goal="Vurdere mobilitet, ADL, fallrisiko og rehabiliteringsmal.",
    backstory="Har et fysioterapeutisk og funksjonsorientert perspektiv.",
    llm=llm,
    verbose=False,
)

kommunikasjonsagent = Agent(
    role="Kommunikasjonsagent",
    goal="Vurdere behov for tilpasset informasjon og kommunikasjon.",
    backstory="Har fokus pa pasientforstaelse, kommunikasjon og oppfolging.",
    llm=llm,
    verbose=False,
)

omsorgsagent = Agent(
    role="Omsorgsagent",
    goal="Vurdere egenomsorg, medikamenthandtering og hjemmeoppfolging.",
    backstory="Har et sykepleiefaglig og kommunalt oppfolgingsperspektiv.",
    llm=llm,
    verbose=False,
)

koordinatoragent = Agent(
    role="Koordinatoragent",
    goal="Samle vurderingene i en strukturert plan med tydelige forbehold.",
    backstory="Har ansvar for samordning og menneske-i-loopen.",
    llm=llm,
    verbose=False,
)


task_funksjon = Task(
    description=f"""
    Les caset:
    {case_reader(case_text)}

    Bruk sjekklisten:
    {rehab_checklist_tool()}

    Lag en kort vurdering av mobilitet, fallrisiko, ADL og rehabiliteringsmal.
    Ikke still diagnose eller gi behandlingsordre.
    """,
    expected_output="Kort funksjonsvurdering med forbehold.",
    agent=funksjonsagent,
)

task_kommunikasjon = Task(
    description=f"""
    Les caset:
    {case_reader(case_text)}

    Vurder behov for tilpasset informasjon, kommunikasjon og oppfolging.
    Ikke anta mer enn det som star i caset.
    """,
    expected_output="Kort vurdering av kommunikasjon og informasjonsbehov.",
    agent=kommunikasjonsagent,
)

task_omsorg = Task(
    description=f"""
    Les caset:
    {case_reader(case_text)}

    Bruk ved behov:
    - {local_guideline_lookup('fallrisiko')}
    - {local_guideline_lookup('egenomsorg')}
    - {local_guideline_lookup('medikamenter')}

    Vurder egenomsorg, medikamenthandtering og behov for hjemmeoppfolging.
    Ikke anta tilgang til andre data enn det som er oppgitt her.
    """,
    expected_output="Kort omsorgsvurdering med praktiske forslag og forbehold.",
    agent=omsorgsagent,
)

task_koordinering = Task(
    description="""
    Samle de tre delvurderingene i en strukturert rehabiliteringsplan.

    Bruk overskriftene:
    - hovedutfordringer
    - foreslatte rehabiliteringsmal
    - foreslatte tiltak
    - hvem som skal involveres
    - hva som ma avklares av menneskelig fagperson
    - forbehold og usikkerhet

    Avslutt med tydelig melding om at dette kun er et undervisningsutkast.
    """,
    expected_output="Samlet rehabiliteringsplan for menneskelig vurdering.",
    agent=koordinatoragent,
)


rehab_crew = Crew(
    agents=[funksjonsagent, kommunikasjonsagent, omsorgsagent, koordinatoragent],
    tasks=[task_funksjon, task_kommunikasjon, task_omsorg, task_koordinering],
    process=Process.sequential,
    verbose=True,
)


resultat = rehab_crew.kickoff()

print("SAMLET RESULTAT")
print(resultat)

print("\nETTERKONTROLL")
pprint(uncertainty_flagger(str(resultat)))


## Hva ser vi i eksemplet?

Selv om eksemplet er lite, illustrerer det flere viktige poeng:

- oppgaven blir delt mellom flere roller i stedet for a ligge hos en enkelt agent
- agentene bruker et avgrenset case og enkle verktoy
- koordinatoren samler delresultatene i en felles struktur
- sluttresultatet presenteres som et utkast, ikke som en automatisk beslutning

Dette er ofte en bedre mate a introdusere multi-agent AI i helse og omsorg pa enn a starte med diagnostikk eller triage. Vi ser hvordan systemet kan organisere arbeid, men vi beholder tydelig menneskelig ansvar.


## Refleksjon

- Hva vant vi pa a dele opp oppgaven i flere agentroller?
- Hvor kunne systemet ha misforstatt eller oversett noe viktig?
- Hvilke deler av resultatet ma alltid vurderes av menneskelig fagperson?
- Nar ville en enklere workflow vaert bedre enn et multi-agent oppsett?
- Hva slags logging og dokumentasjon ville vaert viktig i et mer realistisk system?

Hvis du vil utforske videre, kan du prove a endre caset, legge til en ny agentrolle eller sammenligne dette oppsettet med en enklere en-agent-losning.
